# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laspric/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### Baseline rule

I will prioritize content for review when it has not been updated recently and still has meaningful search visibility. The rule will give a higher score to pages that are older and have more impressions, because these pages may represent stronger opportunities for a content refresh.

The rule uses only information available before the decision. It does not use the decline label, `trend_direction`, or `trend_pct`.

### Reason codes

- `stale_but_visible` — the page has not been updated recently and has meaningful search visibility.
- `stale_low_visibility` — the page is old but has relatively low search visibility.
- `fresh_but_visible` — the page has meaningful search visibility but was updated recently.
- `low_priority` — the page has low search visibility and does not meet the stronger review conditions.

### Intended action

The action label will be:

`review_for_refresh`

for pages that meet the main review condition.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline rule

I will prioritize content that is both stale and visible.

A page receives a higher score when:
- it has not been updated for at least 91 days, and
- it has meaningful search visibility through 90-day impressions.

The score is intentionally simple and transparent. It uses fixed thresholds rather than learned weights so that a non-technical reviewer can understand why a page was ranked highly.

### Reason codes

- `stale_and_visible` — the page is older than 90 days since its last update and has meaningful search impressions.
- `visible_but_not_stale` — the page has meaningful impressions but was updated within the last 90 days.
- `stale_but_low_visibility` — the page is old but has low search visibility.
- `low_priority` — the page is neither sufficiently stale nor sufficiently visible.

### Action labels

- `refresh_review`
- `monitor`

In [1]:
# ============================================================
# ML-07 — Step 2: Build the Transparent Baseline Score
# ============================================================

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# 1. Load the dataset
# ------------------------------------------------------------

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print()


# ------------------------------------------------------------
# 2. Create the baseline signals
# ------------------------------------------------------------

# Signal 1: Staleness
# A page is considered stale if it has not been updated
# for at least 91 days.

df["stale"] = (
    df["days_since_last_update"] >= 91
).astype(int)


# Signal 2: Visibility
# A page is considered visible if it received at least
# 100 impressions during the last 90 days.

df["visible"] = (
    df["impressions_90d"] >= 100
).astype(int)


# ------------------------------------------------------------
# 3. Create the transparent baseline score
# ------------------------------------------------------------

# Score = 1 when BOTH conditions are true:
#   - the page is stale
#   - the page still has meaningful visibility
#
# Otherwise score = 0.

df["score"] = df["stale"] * df["visible"]


# ------------------------------------------------------------
# 4. Assign ONE reason code to every page
# ------------------------------------------------------------

def assign_reason(row):

    if row["stale"] == 1 and row["visible"] == 1:
        return "stale_and_visible"

    elif row["stale"] == 0 and row["visible"] == 1:
        return "visible_but_not_stale"

    elif row["stale"] == 1 and row["visible"] == 0:
        return "stale_but_low_visibility"

    else:
        return "low_priority"


df["reason_code"] = df.apply(assign_reason, axis=1)


# ------------------------------------------------------------
# 5. Assign an action
# ------------------------------------------------------------

# High-priority pages:
#   stale + visible → refresh review
#
# Everything else:
#   monitor

df["action"] = np.where(
    df["score"] == 1,
    "refresh_review",
    "monitor"
)


# ------------------------------------------------------------
# 6. Rank the pages
# ------------------------------------------------------------

# First:
#   score = 1 pages come first
#
# Then:
#   among pages with the same score,
#   pages with more impressions come first.

queue = df.sort_values(
    by=["score", "impressions_90d"],
    ascending=[False, False]
).copy()


# ------------------------------------------------------------
# 7. Keep only the columns required for the queue
# ------------------------------------------------------------

queue = queue[
    [
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d"
    ]
]


# ------------------------------------------------------------
# 8. Create the output folder if necessary
# ------------------------------------------------------------

os.makedirs("work/outputs", exist_ok=True)


# ------------------------------------------------------------
# 9. Save the required CSV
# ------------------------------------------------------------

output_path = "work/outputs/baseline_action_score.csv"

queue.to_csv(
    output_path,
    index=False
)


# ------------------------------------------------------------
# 10. Check the result
# ------------------------------------------------------------

print("Queue shape:", queue.shape)
print()

print("Top 10 ranked pages:")
display(queue.head(10))

print()

print("Reason codes:")
print(queue["reason_code"].value_counts())

print()

print("Actions:")
print(queue["action"].value_counts())

print()

print("Score distribution:")
print(queue["score"].value_counts())

print()

print("Saved to:")
print(output_path)

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

In [2]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir())

Current folder:
/content

Files/folders here:
['.config', 'sample_data']


### Signal 1: Content Staleness

I expect older content that has not been updated for a long time to be more likely to need review. This signal is linked to FlyRank's refresh/staleness logic.

**Verdict: CONFIRMED**

Pages with 91–180 days since their last update had a 61.11% observed decline rate compared with 51.20% for pages updated within 90 days. The 181–365 and 365+ buckets are much smaller, so I would not rely on them alone for a strong conclusion.

### Signal 2: Search Volume

I expect search volume to help identify content with a meaningful opportunity for review. I tested impressions over the trailing 90-day window using volume buckets.

**Verdict: MIXED**

The observed decline rate rises from 38.92% for pages with 0–100 impressions to 60.28% for 101–1k and 62.03% for 1k–10k, but then falls to 52.36% for 10k+ pages. Therefore, volume is useful as an opportunity signal, but the relationship with decline is not consistently increasing.

In [21]:
df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 100, 1000, 10000, float("inf")],
    labels=["0-100", "101-1k", "1k-10k", "10k+"]
)

signal_2 = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("is_declining_label", "size"),
          decline_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

signal_2["decline_rate_pct"] = signal_2["decline_rate"] * 100

print(signal_2)

  volume_bucket     n  decline_rate  decline_rate_pct
0         0-100  8006      0.389208         38.920809
1        101-1k  8485      0.602829         60.282852
2        1k-10k  9907      0.620268         62.026850
3          10k+  3602      0.523598         52.359800


In [ ]:
### Signal 2: Search Volume

I expect search volume to help identify content with a meaningful opportunity for review. I tested impressions over the trailing 90-day window using volume buckets.

**Verdict: MIXED**

The observed decline rate rises from 38.92% for pages with 0–100 impressions to 60.28% for 101–1k and 62.03% for 1k–10k, but then falls to 52.36% for 10k+ pages. Therefore, volume is useful as an opportunity signal, but the relationship with decline is not consistently increasing.

In [17]:
!ls data/raw

content_refresh_anonymized.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.